# DENTEX 4-Class Abnormal Tooth Detection Benchmark (Kaggle)

This notebook runs the unified DENTEX benchmark on Kaggle Notebooks (GPU P100 / T4):
- Automatic `/kaggle/working` directory structure
- Automated dependency setup
- Dataset preparation (Kaggle input dataset or HuggingFace fallback)
- Full model training across Faster R-CNN, DINO, HierarchicalDet
- Evaluation and artifact persistence


## 1. Kaggle Working Directory Setup


In [ ]:
import os
import sys
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working/Dentex')
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
%cd {KAGGLE_WORKING}
if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))
print(f'Active Kaggle directory: {os.getcwd()}')

if not (KAGGLE_WORKING / 'dentex_bench').exists():
    print('Project not found at target directory, cloning repository...')
    !git clone https://github.com/ibrahimethemhamamci/DENTEX.git {KAGGLE_WORKING}


## 2. Auto-Setup Dependencies & Environment Check


In [ ]:
!pip install pycocotools albumentations tabulate
from dentex_bench.env import detect_platform, get_device_info, ensure_dependencies

ensure_dependencies()
dev = get_device_info()
print(f'Platform: {detect_platform()}')
print(f'Hardware: {dev["device"]} ({dev["name"]}), VRAM: {dev["vram_gb"]} GB, Low VRAM: {dev["is_low_vram"]}')


## 3. Data Download & Preparation


In [ ]:
# Check Kaggle input dataset or download from HuggingFace
kaggle_input = Path('/kaggle/input/dentex')
data_flag = f'--data_dir {kaggle_input}' if kaggle_input.exists() else ''
!python scripts/download_and_prepare.py {data_flag} --seed 42


## 4. Train Detectors


In [ ]:
# Train Faster R-CNN, DINO, and HierarchicalDet
!python scripts/train.py --model faster_rcnn --epochs 20
!python scripts/train.py --model dino --epochs 20
!python scripts/train.py --model hierarchical_det --epochs 20


## 5. Evaluate and Package Kaggle Output Artifacts


In [ ]:
!python scripts/evaluate.py --compare_all

from IPython.display import display, Markdown
from pathlib import Path

summary_path = Path('outputs/benchmark_summary.md')
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

print('\n--- Generated Artifacts in /kaggle/working/Dentex/outputs ---')
for p in sorted(Path('outputs').glob('*')):
    print(f' - {p.name} ({p.stat().st_size:,} bytes)')
